# RSNA knee - encode once, then answer the week-3 question

One notebook. Run it with **Save & Run All (Commit)**.

1. Finds everything it needs under `/kaggle/input` by content, and prints what it
   found before doing any work.
2. Times fp16 against bf16 on this GPU. The submission notebook logs
   `native bf16=False` on T4 and then runs `amp bfloat16 (on=True)`; this prices
   what that costs.
3. Encodes training studies to cached features. **Resumable** - shards are
   written every `SHARD` studies, a manifest records what is done, and re-running
   with this notebook's own output attached skips everything already encoded.
   Gold studies go first, so a truncated run still leaves a usable validation set.
4. Trains the classifier head on those features under five label sets and measures
   **how much macro AUC is lost per unit of label corruption**. That slope is the
   only form of the "are better labels worth it" question that 58 gold studies
   can answer.

**This produces no submission and no leaderboard score.** It produces one number,
`slope_per_unit_label_accuracy`, which decides whether the next three weeks go
into labels or into the efficiency prize.

**Attach:** the competition data, any dataset containing
`raptor_ft_coatnet_v5_full_swa.pt`, and `weak_labels.csv` uploaded as a dataset.
Add this notebook's own previous output to resume.


In [ ]:
# =============================================================== config
MAX_STUDIES   = 4407     # set to 0 to skip encoding and only re-run the gate
TIME_BUDGET_H = 8.0      # exit cleanly and save well before the session limit
SHARD         = 250
RUN_GATE      = True
SEED          = 0

# gate hyperparameters
EPOCHS, LR, WD, DROP, N_SEEDS, HOLDOUT = 60, 3e-4, 1e-2, 0.3, 3, 0.15

ARM = dict(file="raptor_ft_coatnet_v5_full_swa.pt",
           arch="coatnet_rmlp_2_rw_384.sw_in12k_ft_in1k", res=384, img=336,
           span=(0.02, 0.98), k_eval=62)

import os, glob, time, gc, json, pathlib, shutil
os.environ.setdefault("HF_HUB_OFFLINE", "1")
os.environ.setdefault("TRANSFORMERS_OFFLINE", "1")
import numpy as np, pandas as pd, torch, torch.nn as nn, torch.nn.functional as F
import timm
torch.backends.cudnn.benchmark = True
torch.backends.cuda.matmul.allow_tf32 = True
torch.manual_seed(SEED); np.random.seed(SEED)

DEV = "cuda" if torch.cuda.is_available() else "cpu"
LABELS = ["ACL","MCL","Medial Meniscus","Lateral Meniscus","Medial OA","Lateral OA",
          "PF OA","Effusion","Synovitis","Baker's","Contusion","Fracture"]
OUT = pathlib.Path("/kaggle/working/feat"); OUT.mkdir(parents=True, exist_ok=True)

if DEV == "cuda":
    for i in range(torch.cuda.device_count()):
        p = torch.cuda.get_device_properties(i)
        cc = (p.major, p.minor)
        print(f"gpu{i}: {p.name} sm_{p.major}{p.minor} | {p.total_mem_gb if False else p.total_memory/2**30:.0f} GiB"
              f" | native bf16 = {cc >= (8, 0)}")
    print("  NB: memory is PER CARD and does not pool across the two.")
print("torch", torch.__version__, "| timm", timm.__version__, "| device", DEV)


In [ ]:
# ================================================= auto-discovery
# Finds inputs by content, not by dataset name, so it does not matter what the
# attached datasets are called.
#
# CRITICAL: never let a recursive glob loose on /kaggle/input. The competition
# dataset is 819,640 DICOM files over 574 GB, and `glob("**", recursive=True)`
# walks every one of them before returning. Every search here prunes the image
# trees and bounds its depth.
SKIP_DIRS = {"train_series", "test_series", "train_images", "test_images",
             "__pycache__", ".git"}
MAX_DEPTH = 5

def _scan(root, depth=MAX_DEPTH):
    """Yield (path, size) for files under root, pruning DICOM trees."""
    root = root.rstrip("/")
    base = root.count(os.sep)
    for d, dirs, files in os.walk(root):
        dirs[:] = [x for x in dirs if x not in SKIP_DIRS]
        if d.count(os.sep) - base >= depth:
            dirs[:] = []
        for f in files:
            fp = os.path.join(d, f)
            try:
                yield fp, os.path.getsize(fp)
            except OSError:
                pass

def _roots():
    out = []
    for r in ("/kaggle/input", "/kaggle/working"):
        if os.path.isdir(r):
            out += [os.path.join(r, x) for x in sorted(os.listdir(r))]
    return [x for x in out if os.path.isdir(x)]

t_scan = time.time()
INVENTORY = {}
for r in _roots():
    INVENTORY[r] = list(_scan(r))
print(f"scanned {len(INVENTORY)} attached source(s) in {time.time()-t_scan:.1f}s "
      f"({sum(len(v) for v in INVENTORY.values())} files, DICOM trees pruned)\n")

def find_file(name):
    hits = [p for v in INVENTORY.values() for p, _ in v if os.path.basename(p) == name]
    return sorted(hits)

def find_glob(pat):
    import fnmatch
    return sorted(p for v in INVENTORY.values() for p, _ in v
                  if fnmatch.fnmatch(os.path.basename(p), pat))

def discover():
    d = {}
    comp = None
    for cand in ["/kaggle/input/rsna-knee-abnormality-detection",
                 "/kaggle/input/competitions/rsna-knee-abnormality-detection"]:
        if os.path.exists(cand + "/train.csv"):
            comp = cand
            break
    if comp is None:
        for p in find_file("train.csv"):
            if os.path.isdir(os.path.dirname(p) + "/train_series"):
                comp = os.path.dirname(p)
                break
    d["competition"] = comp
    w = find_file(ARM["file"])
    d["weights"] = w[0] if w else None
    lab = find_file("weak_labels.csv")
    d["weak_labels"] = lab[0] if lab else None
    d["shards"] = find_glob("shard_*.npz")
    # Prebuilt volume corpora. dreaddevelopment/knee-raptor-corpus holds
    # all_vols.npy at exactly 2200 x 64 x 336 x 336 uint8 -- arm 0's build_study
    # output, already decoded. -ext adds ~830 more. Using them skips the 1.89s
    # of DICOM I/O per study and leaves only the 0.99s encode.
    d["corpora"] = []
    for vol, ids, msk in (("all_vols.npy", "all_ids.npy", "all_masks.npy"),
                          ("extra_vols.npy", "extra_ids.npy", "extra_masks.npy")):
        v = find_file(vol)
        if v:
            i, m = find_file(ids), find_file(msk)
            if i and m:
                d["corpora"].append((v[0], i[0], m[0]))
    return d

D = discover()
print("=== discovered ===")
for k in ("competition", "weights", "weak_labels"):
    print(f"  {k:<14} {D[k] or 'NOT FOUND'}")
print(f"  {'shards':<14} {len(D['shards'])} file(s)")
print(f"  {'corpora':<14} {len(D['corpora'])} prebuilt volume set(s)")

missing = [k for k in ("competition", "weights") if not D[k]]
if RUN_GATE and not D["weak_labels"]:
    missing.append("weak_labels")
if missing:
    print("\n  attached sources and their small files:")
    for r, v in INVENTORY.items():
        small = [os.path.basename(p) for p, s in v if s < 5e8][:8]
        print(f"    {r}\n      {small}")
    raise RuntimeError(f"missing: {missing}. Attach the dataset(s) that contain them.")

ROOT, TSDIR = D["competition"], D["competition"] + "/train_series"
train = pd.read_csv(ROOT + "/train.csv")
train["StudyInstanceUID"] = train["StudyInstanceUID"].astype(str)
tser = pd.read_csv(ROOT + "/train_series.csv")
for col in ("StudyInstanceUID", "SeriesInstanceUID"):
    tser[col] = tser[col].astype(str)
SERIES = {k: v.to_dict("records") for k, v in tser.groupby("StudyInstanceUID")}
gold_mask = train[LABELS].notna().all(axis=1).to_numpy()
rng = np.random.default_rng(SEED)
rest = train.loc[~gold_mask, "StudyInstanceUID"].tolist(); rng.shuffle(rest)
ORDER = train.loc[gold_mask, "StudyInstanceUID"].tolist() + rest
print(f"\n{len(train)} studies, {int(gold_mask.sum())} gold (encoded first), "
      f"{len(tser)} series")


## What else is attached

One of the datasets in play is called *RSNA Knee LLM-read reports*. If it holds
LLM-derived labels for the 4,407 training studies, that is exactly what week 3
was going to build, and it turns the gate from a simulated perturbation into a
direct test: does a genuinely different labeler beat the regex lexicon?

This cell inspects every small CSV/parquet in the attached datasets, reports the
schema of anything that looks like a per-study label table, and adds it to the
gate as an extra arm. It never fails the run - if nothing is found, the gate
proceeds with the arms it already has.


In [ ]:
# ============================== inspect attached tables, harvest label sets
def _norm(c):
    return "".join(ch for ch in str(c).lower() if ch.isalnum())

_LAB_N = {_norm(t): t for t in LABELS}
COMP_FILES = {"train.csv", "test.csv", "train_series.csv", "test_series.csv",
              "sample_submission.csv", "weak_labels.csv"}

CANDIDATES = {}
print("=== per-study tables found in the attached datasets ===")
for r, files in INVENTORY.items():
    for fp, sz in files:
        base = os.path.basename(fp)
        if base in COMP_FILES or sz > 200e6:
            continue
        if not base.lower().endswith((".csv", ".parquet")):
            continue
        try:
            head = (pd.read_parquet(fp) if base.lower().endswith(".parquet")
                    else pd.read_csv(fp, nrows=5))
        except Exception:
            continue
        cols = list(head.columns)
        uid = next((c for c in cols if _norm(c) in
                    ("studyinstanceuid", "studyuid", "study", "id")), None)
        hit = {_LAB_N[_norm(c)]: c for c in cols if _norm(c) in _LAB_N}
        if uid and len(hit) >= 6:
            CANDIDATES[fp] = (uid, hit)
            print(f"  {fp}")
            print(f"     uid={uid!r}  {len(hit)}/12 labels  rows~{sz/1e6:.1f}MB")
        elif uid:
            print(f"  {fp}  (uid found, only {len(hit)}/12 label columns: "
                  f"{cols[:8]}{'...' if len(cols) > 8 else ''})")
if not CANDIDATES:
    print("  none with >=6 of the 12 label columns.")
    print("  If a dataset holds LLM labels under different column names, print")
    print("  its columns above and map them by hand.")


In [ ]:
# ============================ pipeline, verbatim from the submission notebook
# Any edit here makes the cached features stop matching what the submission
# pipeline produces at test time. Do not touch.
IMG = int(ARM["img"])
CROP_MM = 140.0
SPAN_LO, SPAN_HI = ARM["span"]
SLOTS = [("Sagittal", 1, 18), ("Sagittal", 0, 14),
         ("Coronal", 1, 12), ("Coronal", 0, 8), ("Axial", -1, 12)]
MAXS = sum(s[2] for s in SLOTS)
K_EVAL = int(ARM["k_eval"])
NORM = "imagenet"
_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
_STD = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

def build_backbone(arch, pretrained=False):
    hybrid = arch.startswith(('maxvit', 'maxxvit', 'coatnet', 'coat_', 'convnext'))
    is_vit = not hybrid and any((k in arch for k in ('vit', 'deit', 'dinov2', 'eva', 'beit')))
    kw = dict(pretrained=pretrained, num_classes=0, in_chans=3)
    if is_vit:
        kw.update(global_pool='token', dynamic_img_size=True)
    else:
        kw.update(global_pool='avg')
    return timm.create_model(arch, **kw)

class RaptorClassifier(nn.Module):

    def __init__(self, backbone, F_dim=768, n=12, drop=0.2):
        super().__init__()
        self.backbone = backbone
        self.norm = nn.LayerNorm(F_dim)
        self.att = nn.Sequential(nn.Linear(F_dim, 256), nn.Tanh(), nn.Dropout(drop), nn.Linear(256, n))
        self.clsW = nn.Parameter(torch.zeros(n, F_dim))
        self.clsb = nn.Parameter(torch.zeros(n))
        nn.init.trunc_normal_(self.clsW, std=0.02)
        self.n = n

    def encode(self, x):
        B, K = x.shape[:2]
        f = self.backbone(x.flatten(0, 1))
        return f.view(B, K, -1)

    def head(self, feats):
        h = self.norm(feats)
        a = self.att(h)
        a = torch.softmax(a, dim=1)
        pooled = torch.einsum('bkn,bkf->bnf', a, h)
        logits = (pooled * self.clsW).sum(-1) + self.clsb
        return logits

    def forward(self, x):
        return self.head(self.encode(x))

def load_model(pt_path, arch_default, res_default, device, ngpu=1):
    ck = torch.load(pt_path, map_location='cpu', weights_only=False)
    arch = ck.get('arch', arch_default)
    ck_res = int(ck.get('res', res_default))
    bb = build_backbone(arch, pretrained=False)
    model = RaptorClassifier(bb, F_dim=bb.num_features)
    model.load_state_dict(ck['model'], strict=True)
    model.eval().to(device)
    del ck
    gc.collect()
    return (model, ck_res)

def _eval_centers(mask, D, k):
    valid = np.where(mask > 0)[0]
    if len(valid) < 3:
        valid = np.arange(min(3, D))
    lo, hi = (int(valid.min()), int(valid.max()))
    cs = [c for c in range(lo + 1, hi) if c - 1 >= lo and c + 1 <= hi]
    if not cs:
        cs = [max(1, min((lo + hi) // 2, D - 2))]
    idx = np.linspace(0, len(cs) - 1, k).round().astype(int)
    return [cs[i] for i in idx]

def eval_windows(vol, mask, k, res, norm=NORM):
    D = vol.shape[0]
    cs = _eval_centers(mask, D, k)
    wins = np.empty((len(cs), 3, res, res), np.float32)
    for j, c in enumerate(cs):
        c = max(1, min(c, D - 2))
        tri = np.stack([vol[c - 1], vol[c], vol[c + 1]], 0).astype(np.float32) / 255.0
        t = torch.from_numpy(tri)
        if t.shape[-1] != res:
            t = F.interpolate(t[None], size=(res, res), mode='bilinear', align_corners=False)[0]
        wins[j] = t.numpy()
    x = torch.from_numpy(wins)
    if norm == 'imagenet':
        x = (x - _MEAN) / _STD
    return x

def _make_reader():
    import pydicom, cv2
    from pydicom.pixel_data_handlers.util import apply_modality_lut
    _hdr_cache, _px_cache = {}, {}

    def order_and_meta(sdir):
        if sdir in _hdr_cache:
            return _hdr_cache[sdir]
        fs = glob.glob(sdir + '/*.dcm')
        recs = []
        ps_list = []
        for f in fs:
            try:
                h = pydicom.dcmread(f, stop_before_pixels=True)
                iop = getattr(h, 'ImageOrientationPatient', None)
                ipp = getattr(h, 'ImagePositionPatient', None)
                if iop is not None and ipp is not None and (len(iop) == 6):
                    r = np.array(iop[:3], float)
                    c = np.array(iop[3:], float)
                    n = np.cross(r, c)
                    pos = float(np.dot(np.array(ipp, float), n))
                else:
                    pos = float(getattr(h, 'InstanceNumber', 0) or 0)
                ps = getattr(h, 'PixelSpacing', None)
                ps = float(ps[0]) if ps is not None else 0.5
                ps_list.append(ps)
                recs.append((pos, f, ps))
            except Exception:
                recs.append((0.0, f, 0.5))
        recs.sort(key=lambda x: x[0])
        med_ps = float(np.median(ps_list)) if ps_list else 0.5
        out = ([(f, ps) for _, f, ps in recs], med_ps)
        _hdr_cache[sdir] = out
        return out

    def read_px(f):
        a = _px_cache.get(f)
        if a is not None:
            return a
        d = pydicom.dcmread(f)
        a = apply_modality_lut(d.pixel_array, d).astype(np.float32)
        if str(getattr(d, 'PhotometricInterpretation', '')) == 'MONOCHROME1':
            a = a.max() - a
        _px_cache[f] = a
        return a

    def mm_crop_resize(a, ps):
        h, w = a.shape
        cpx = int(round(CROP_MM / max(ps, 0.001)))
        cpx = min(cpx, min(h, w))
        y0 = (h - cpx) // 2
        x0 = (w - cpx) // 2
        a = a[y0:y0 + cpx, x0:x0 + cpx]
        return cv2.resize(a, (IMG, IMG), interpolation=cv2.INTER_AREA)

    def reset_cache():
        _hdr_cache.clear()
        _px_cache.clear()
    return (order_and_meta, read_px, mm_crop_resize, reset_cache)

def _pick_series_for_slot(rows, plane, fluid, used):
    cands = [r for r in rows if r['Anatomical_Plane'] == plane and r['SeriesInstanceUID'] not in used]
    if fluid in (0, 1):
        pref = [r for r in cands if int(r.get('Fluid_Sensitive', 0) or 0) == fluid]
        if pref:
            return pref[0]
    return cands[0] if cands else None

def build_study(sid, ser_records, tsdir, reader):
    order_and_meta, read_px, mm_crop_resize = reader[0], reader[1], reader[2]
    rows = ser_records.get(sid, [])
    vol = np.zeros((MAXS, IMG, IMG), np.uint8)
    idx = 0
    used = set()
    for plane, fluid, k in SLOTS:
        r = _pick_series_for_slot(rows, plane, fluid, used)
        if r is None:
            idx += k
            continue
        used.add(r['SeriesInstanceUID'])
        files, med_ps = order_and_meta(f"{tsdir}/{sid}/{r['SeriesInstanceUID']}")
        if not files:
            idx += k
            continue
        n = len(files)
        lo, hi = (int(n * SPAN_LO), int(n * SPAN_HI) - 1)
        hi = max(hi, lo)
        picks = np.linspace(lo, hi, k).round().astype(int) if n > 1 else [0] * k
        arrs = []
        pss = []
        for p in picks:
            fp, ps = files[min(p, n - 1)]
            try:
                arrs.append(read_px(fp))
                pss.append(ps)
            except Exception:
                arrs.append(None)
                pss.append(med_ps)
        valid = [a for a in arrs if a is not None]
        if valid:
            allpx = np.concatenate([a.ravel() for a in valid])
            loq, hiq = np.percentile(allpx, [2.0, 98.0])
        else:
            loq, hiq = (0.0, 1.0)
        for a, ps in zip(arrs, pss):
            if idx >= MAXS:
                break
            if a is None:
                idx += 1
                continue
            aw = np.clip((a - loq) / (hiq - loq + 1e-06), 0, 1)
            aw = mm_crop_resize(aw, ps if ps > 0 else med_ps)
            vol[idx] = (aw * 255).astype(np.uint8)
            idx += 1
        if idx >= MAXS:
            break
    mask = (vol.reshape(MAXS, -1).sum(1) > 0).astype(np.uint8)
    return (vol, mask)


In [ ]:
# ================================================ encoder + timing probe
model, RES = load_model(D["weights"], ARM["arch"], ARM["res"], DEV)
F_DIM = model.norm.normalized_shape[0]
print(f"encoder | res {RES} | F_dim {F_DIM} | "
      f"{sum(p.numel() for p in model.parameters())/1e6:.1f}M params")
reader = _make_reader()

@torch.no_grad()
def encode_windows(win, dtype=torch.float16):
    x = win.unsqueeze(0).to(DEV, non_blocking=True)
    if DEV == "cpu" or dtype is None:
        return model.encode(x)[0]
    with torch.autocast("cuda", dtype=dtype):
        return model.encode(x)[0]

probe = [s for s in ORDER[:6] if s in SERIES]
t = time.time(); vols = []
for sid in probe:
    v, k = build_study(sid, SERIES, TSDIR, reader)
    vols.append(eval_windows(v, k, k=K_EVAL, res=RES, norm=NORM)); reader[3]()
t_build = (time.time() - t) / max(len(probe), 1)

def bench(dt, reps=3):
    if DEV == "cpu": return float("nan")
    for w in vols[:2]: encode_windows(w, dt)
    torch.cuda.synchronize(); t0 = time.time()
    for _ in range(reps):
        for w in vols: encode_windows(w, dt)
    torch.cuda.synchronize()
    return (time.time() - t0) / (reps * len(vols))

TIMING = {"build_s": t_build}
for nm, dt in (("fp16", torch.float16), ("bf16", torch.bfloat16), ("fp32", None)):
    try: TIMING[nm] = bench(dt)
    except Exception as e: TIMING[nm] = float("nan"); print(f"  {nm}: {type(e).__name__}")
print("\n=== per study, one arm ===")
print(f"  build_study + eval_windows : {TIMING['build_s']:.2f}s   (I/O bound)")
for nm in ("fp16", "bf16", "fp32"):
    print(f"  encode {nm:<4}                 : {TIMING[nm]:.2f}s")
if np.isfinite(TIMING["fp16"]) and np.isfinite(TIMING["bf16"]) and TIMING["fp16"] > 0:
    r = TIMING["bf16"] / TIMING["fp16"]
    print(f"\n  bf16 / fp16 = {r:.2f}x  ->  " +
          ("set AMP_PREF='auto' in cell 23 of the submission notebook"
           if r > 1.08 else "no meaningful difference, leave AMP_PREF alone"))
per_study = TIMING["build_s"] + (TIMING["fp16"] if np.isfinite(TIMING["fp16"]) else 0)
print(f"\n  encode cost {per_study:.2f}s/study -> "
      f"{per_study*len(ORDER)/3600:.1f}h for all {len(ORDER)}, "
      f"{per_study*1500/3600:.1f}h for 1500")
json.dump(TIMING, open("/kaggle/working/timing_probe.json", "w"), indent=1)


In [ ]:
# ================================ prebuilt volumes: map, then VERIFY
# Never trust someone else's corpus without checking it against our own
# build_study on the same study. If it does not match bit for bit we fall back
# to decoding DICOMs, because features built from a different preprocessing
# would silently poison every downstream comparison.
CORPUS = {}
for vpath, ipath, mpath in D.get("corpora", []):
    try:
        vols = np.load(vpath, mmap_mode="r")
        ids = np.load(ipath, allow_pickle=True)
        masks = np.load(mpath, mmap_mode="r")
        ids = np.array([str(x) for x in np.asarray(ids).ravel()])
        if vols.shape[0] != len(ids):
            print(f"  {os.path.basename(vpath)}: {vols.shape[0]} vols vs {len(ids)} ids, skipping")
            continue
        print(f"  {os.path.basename(vpath)}: {vols.shape} {vols.dtype}, {len(ids)} ids")
        if tuple(vols.shape[1:]) != (MAXS, IMG, IMG):
            print(f"     shape is not ({MAXS}, {IMG}, {IMG}) -- skipping, wrong arm")
            continue
        for r, sid in enumerate(ids):
            CORPUS.setdefault(sid, (vols, masks, r))
    except Exception as e:
        print(f"  {vpath}: {type(e).__name__}: {e}")

CORPUS_OK = False
if CORPUS:
    check = [s for s in ORDER if s in CORPUS and s in SERIES][:3]
    print(f"\nverifying {len(check)} study(ies) against our own build_study")
    diffs = []
    for sid in check:
        mine, mymask = build_study(sid, SERIES, TSDIR, reader); reader[3]()
        vols, masks, r = CORPUS[sid]
        theirs = np.asarray(vols[r])
        d = float(np.abs(mine.astype(np.int16) - theirs.astype(np.int16)).max())
        diffs.append(d)
        print(f"  {sid[:26]}  max abs diff {d:.0f}  "
              f"mask match {bool((np.asarray(masks[r]) == mymask).all())}")
    CORPUS_OK = bool(diffs) and max(diffs) <= 1     # allow 1 LSB of rounding
    print(f"\n  corpus {'VERIFIED -- using it, skips ~1.9s of I/O per study' if CORPUS_OK else 'REJECTED -- decoding DICOMs instead'}")
    if not CORPUS_OK:
        CORPUS = {}
        print("  (a mismatch means their preprocessing differs from arm 0's;")
        print("   features built from it would not match the submission pipeline)")
n_from_corpus = len([s for s in ORDER if s in CORPUS])
if CORPUS:
    est = n_from_corpus * TIMING["fp16"] + (len(ORDER) - n_from_corpus) * per_study
    print(f"\n  {n_from_corpus} of {len(ORDER)} studies come from the corpus")
    print(f"  estimated total {est/3600:.1f}h, against "
          f"{per_study*len(ORDER)/3600:.1f}h decoding everything")


In [ ]:
# ============================================== encode, resumable
MANIFEST = pathlib.Path("/kaggle/working/manifest.json")
done = set()
for p in D["shards"] + sorted(glob.glob(str(OUT / "shard_*.npz"))):
    try: done |= {str(s) for s in np.load(p, allow_pickle=True)["ids"]}
    except Exception as e: print(f"  unreadable shard {p}: {type(e).__name__}")
print(f"already encoded: {len(done)}")

todo = [s for s in ORDER if s not in done and s in SERIES][:max(MAX_STUDIES, 0)]
print(f"to encode this run: {len(todo)}"
      f"  (~{per_study*len(todo)/3600:.1f}h at the probed rate)")

t0 = time.time(); buf_f, buf_i, failed = [], [], []
shard = len(glob.glob(str(OUT / "shard_*.npz")))

def flush():
    global buf_f, buf_i, shard
    if not buf_f: return
    # NOTE: np.savez_compressed APPENDS ".npz" unless the name already ends in it.
    # A ".npz.tmp" name silently becomes ".npz.tmp.npz" and the replace below
    # then fails on a path that was never written. Keep .npz last.
    tmp = OUT / f"shard_{shard:04d}.tmp.npz"
    np.savez_compressed(tmp, feats=np.stack(buf_f).astype(np.float16),
                        ids=np.array(buf_i, dtype=object))
    assert tmp.exists(), f"savez wrote something other than {tmp}"
    os.replace(tmp, OUT / f"shard_{shard:04d}.npz")   # atomic: never a half shard
    done.update(buf_i)
    MANIFEST.write_text(json.dumps(
        {"n_done": len(done), "shards": shard + 1, "F_dim": int(F_DIM),
         "arm": ARM["file"], "failed": len(failed)}, indent=1))
    print(f"  wrote shard_{shard:04d}.npz ({len(buf_i)}), total {len(done)}", flush=True)
    shard += 1; buf_f, buf_i = [], []

for i, sid in enumerate(todo):
    if time.time() - t0 > TIME_BUDGET_H * 3600:
        print(f"time budget reached at {i}/{len(todo)} -- rerun with this "
              f"notebook's output attached to continue"); break
    try:
        if sid in CORPUS:
            vols, masks, r = CORPUS[sid]
            vol, msk = np.asarray(vols[r]), np.asarray(masks[r])
        else:
            vol, msk = build_study(sid, SERIES, TSDIR, reader)
        win = eval_windows(vol, msk, k=K_EVAL, res=RES, norm=NORM)
        buf_f.append(encode_windows(win).float().cpu().numpy()); buf_i.append(sid)
        del vol, msk, win
    except Exception as e:
        failed.append((sid, f"{type(e).__name__}: {e}"))
    reader[3]()
    if len(buf_f) >= SHARD:
        flush(); gc.collect()
    if (i + 1) % 200 == 0:
        el = time.time() - t0
        print(f"  {i+1}/{len(todo)} | {el/60:.0f}m | {el/(i+1):.2f}s/study | "
              f"eta {el/(i+1)*(len(todo)-i-1)/60:.0f}m", flush=True)
flush()
print(f"\nencoded total {len(done)} | failures {len(failed)}")
for s, e in failed[:10]: print("   ", s[:24], e)
print(f"cache {sum(p.stat().st_size for p in OUT.glob('*.npz'))/1e6:.0f} MB")
del vols; gc.collect()
if DEV == "cuda": torch.cuda.empty_cache()


In [ ]:
# ================================================== load cache for the gate
assert RUN_GATE, "RUN_GATE is False"
shards = sorted(set(D["shards"] + glob.glob(str(OUT / "shard_*.npz"))))
Fl, il = [], []
for p in shards:
    z = np.load(p, allow_pickle=True); Fl.append(z["feats"]); il += [str(s) for s in z["ids"]]
FEATS = np.concatenate(Fl).astype(np.float32); IDS = np.array(il); del Fl
_, keep = np.unique(IDS, return_index=True); keep = np.sort(keep)
if len(keep) < len(IDS):
    print(f"dropping {len(IDS)-len(keep)} duplicates"); FEATS, IDS = FEATS[keep], IDS[keep]
print(f"features {FEATS.shape} ({FEATS.nbytes/1e6:.0f} MB)")

wl = pd.read_csv(D["weak_labels"]); wl["StudyInstanceUID"] = wl["StudyInstanceUID"].astype(str)
wl = wl.set_index("StudyInstanceUID").loc[IDS].reset_index()
Y = {k: wl[[f"{k}::{t}" for t in LABELS]].to_numpy(np.float32) for k in ("raw","fix","cal")}
GOLD = wl[[f"gold::{t}" for t in LABELS]].to_numpy(np.float32)
is_gold = wl["is_gold"].to_numpy().astype(bool)
print(f"gold in cache {int(is_gold.sum())} | weakly labelled {int((~is_gold).sum())}")
assert is_gold.sum() >= 40, "too few gold studies encoded; rerun encoding"
assert (~is_gold).sum() >= 300, "too few training studies; rerun encoding"
F_DIM = FEATS.shape[-1]


In [ ]:
# ================================================== metrics, head, arms
def auc(y, p):
    y = np.asarray(y, float); p = np.asarray(p, float)
    ok = np.isfinite(y) & np.isfinite(p); y, p = y[ok], p[ok]
    n1, n0 = int((y == 1).sum()), int((y == 0).sum())
    if n1 == 0 or n0 == 0: return np.nan
    r = pd.Series(p).rank().to_numpy()
    return (r[y == 1].sum() - n1*(n1+1)/2) / (n1*n0)

def macro(y, p): return float(np.nanmean([auc(y[:, j], p[:, j]) for j in range(y.shape[1])]))

def boot(y, p, n=2000, seed=0):
    rng = np.random.default_rng(seed); N = len(y); o = []
    for _ in range(n):
        i = rng.integers(0, N, N); v = macro(y[i], p[i])
        if np.isfinite(v): o.append(v)
    return float(np.percentile(o, 2.5)), float(np.percentile(o, 97.5))

class Head(nn.Module):
    """RaptorClassifier's head verbatim, backbone removed."""
    def __init__(self, F_dim, n=12, drop=DROP):
        super().__init__()
        self.norm = nn.LayerNorm(F_dim)
        self.att = nn.Sequential(nn.Linear(F_dim, 256), nn.Tanh(),
                                 nn.Dropout(drop), nn.Linear(256, n))
        self.clsW = nn.Parameter(torch.zeros(n, F_dim)); self.clsb = nn.Parameter(torch.zeros(n))
        nn.init.trunc_normal_(self.clsW, std=0.02)
    def forward(self, f):
        h = self.norm(f); a = torch.softmax(self.att(h), dim=1)
        return (torch.einsum("bkn,bkf->bnf", a, h) * self.clsW).sum(-1) + self.clsb

@torch.no_grad()
def predict(head, X, bs=64):
    head.eval()
    return np.concatenate([torch.sigmoid(head(torch.from_numpy(X[i:i+bs]).to(DEV)).float())
                           .cpu().numpy() for i in range(0, len(X), bs)])

def corrupt(Yv, p, seed):
    """Resample a fraction p of cells from each column's own marginal. Permuting
    within a column keeps the base rate exactly, so the arms differ in label
    ACCURACY only -- otherwise this would confound accuracy with prevalence."""
    g = np.random.default_rng(seed); o = Yv.copy()
    for j in range(Yv.shape[1]):
        hit = g.random(len(Yv)) < p
        o[hit, j] = Yv[g.permutation(len(Yv))[hit], j]
    return o

ARMS = {"fix": Y["fix"], "noise10": corrupt(Y["fix"], 0.10, 101),
        "noise25": corrupt(Y["fix"], 0.25, 102), "raw": Y["raw"], "cal": Y["cal"]}

# Any harvested per-study label table becomes an extra arm. Missing labels and
# missing studies fall back to `fix`, so the arm differs from `fix` ONLY where
# the other labeler actually has an opinion -- otherwise the comparison would
# confound label quality with coverage.
for fp, (uid, hit) in CANDIDATES.items():
    try:
        tbl = (pd.read_parquet(fp) if fp.lower().endswith(".parquet") else pd.read_csv(fp))
        tbl[uid] = tbl[uid].astype(str)
        tbl = tbl.drop_duplicates(subset=[uid]).set_index(uid)
        A = Y["fix"].copy()
        n_cell = 0
        for j, t in enumerate(LABELS):
            if t not in hit:
                continue
            v = pd.to_numeric(tbl[hit[t]], errors="coerce").reindex(IDS)
            ok = v.notna().to_numpy()
            A[ok, j] = np.clip(v.to_numpy()[ok], 0.0, 1.0).astype(np.float32)
            n_cell += int(ok.sum())
        cover = n_cell / A.size
        name = "llm_" + pathlib.Path(fp).stem[:14]
        if cover < 0.05:
            print(f"  skipping {name}: covers only {cover:.1%} of cells")
            continue
        ARMS[name] = A
        agree = float((( A[is_gold] > 0.5) == (GOLD[is_gold] > 0.5)).mean())
        print(f"  arm {name}: {len(hit)}/12 labels, {cover:.0%} of cells, "
              f"gold agreement {agree:.3f} (lexicon fix = "
              f"{float(((Y['fix'][is_gold] > 0.5) == (GOLD[is_gold] > 0.5)).mean()):.3f})")
    except Exception as e:
        print(f"  arm from {fp} failed: {type(e).__name__}: {e}")
# One epoch-selection target for every arm. `cal` is a monotone map of `fix`, so
# binarising each arm's own labels would early-stop them against different
# targets and void the comparison.
VAL_TARGET = (Y["fix"] > 0.5).astype(np.float32)

ckpt_pred = None
try:
    ck = torch.load(D["weights"], map_location="cpu", weights_only=False)
    st = ck.get("model", ck)
    h0 = Head(F_DIM, drop=0.0)
    miss, _ = h0.load_state_dict({k: v for k, v in st.items()
                                  if not k.startswith("backbone.")}, strict=False)
    if miss: print("  WARNING missing head keys:", miss)
    ckpt_pred = predict(h0.to(DEV).eval(), FEATS[is_gold])
    print(f"shipped head on gold: macro {macro(GOLD[is_gold], ckpt_pred):.4f}")
    del ck, st, h0
except Exception as e:
    print("ckpt head baseline skipped:", type(e).__name__, e)


In [ ]:
# ============================================================ train the arms
def train_one(Ytr, seed):
    g = np.random.default_rng(seed); idx = np.where(~is_gold)[0]; g.shuffle(idx)
    k = int(len(idx)*HOLDOUT); va, tr = idx[:k], idx[k:]
    Xtr = torch.from_numpy(FEATS[tr]).to(DEV); Ttr = torch.from_numpy(Ytr[tr]).to(DEV)
    torch.manual_seed(seed)
    head = Head(F_DIM).to(DEV)
    opt = torch.optim.AdamW(head.parameters(), lr=LR, weight_decay=WD)
    sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, EPOCHS)
    lf = nn.BCEWithLogitsLoss(); best, bstate = -1.0, None
    for ep in range(EPOCHS):
        head.train(); perm = torch.randperm(len(tr), device=DEV)
        for i in range(0, len(tr), 32):
            b = perm[i:i+32]; opt.zero_grad(set_to_none=True)
            lf(head(Xtr[b]), Ttr[b]).backward(); opt.step()
        sch.step()
        mm = macro(VAL_TARGET[va], predict(head, FEATS[va]))   # never gold
        if mm > best:
            best, bstate = mm, {k2: v.detach().clone() for k2, v in head.state_dict().items()}
    head.load_state_dict(bstate); del Xtr, Ttr
    if DEV == "cuda": torch.cuda.empty_cache()
    return head, best

RESULTS = {}
for name, Yarm in ARMS.items():
    t0 = time.time(); preds, sel = [], []
    for s in range(N_SEEDS):
        h, b = train_one(Yarm, SEED + s); preds.append(predict(h, FEATS[is_gold]))
        sel.append(b); del h
    P = np.mean(preds, 0)
    RESULTS[name] = dict(pred=P, macro=macro(GOLD[is_gold], P),
                         holdout=float(np.mean(sel)), secs=time.time()-t0)
    print(f"{name:>8}: gold macro {RESULTS[name]['macro']:.4f} | "
          f"weak-holdout {RESULTS[name]['holdout']:.4f} | {RESULTS[name]['secs']:.0f}s",
          flush=True)


In [ ]:
# ================================================================== report
yg = GOLD[is_gold]
print(f"=== gold macro AUC, n={len(yg)}, {N_SEEDS} seeds averaged ===\n")
print(f"{'arm':>8}{'macro':>9}{'95% CI':>20}{'vs fix':>10}")
if ckpt_pred is not None:
    lo, hi = boot(yg, ckpt_pred)
    print(f"{'ckpt':>8}{macro(yg, ckpt_pred):>9.4f}{f'[{lo:.3f}, {hi:.3f}]':>20}{'-':>10}")
base = RESULTS["fix"]["macro"]
for n in RESULTS:
    lo, hi = boot(yg, RESULTS[n]["pred"])
    d = "-" if n == "fix" else f"{RESULTS[n]['macro']-base:+.4f}"
    print(f"{n:>8}{RESULTS[n]['macro']:>9.4f}{f'[{lo:.3f}, {hi:.3f}]':>20}{d:>10}")

def paired(a, b, n=2000, seed=1):
    """Same resampled studies both sides. Comparing two independent intervals is
    the wrong test for a paired comparison and is far too conservative."""
    rng = np.random.default_rng(seed); N = len(yg); d = []
    for _ in range(n):
        i = rng.integers(0, N, N); x, y = macro(yg[i], a[i]), macro(yg[i], b[i])
        if np.isfinite(x) and np.isfinite(y): d.append(x - y)
    d = np.array(d)
    return d.mean(), np.percentile(d, 2.5), np.percentile(d, 97.5), float((d < 0).mean())

print("\n=== paired bootstrap against fix ===")
sl = {}
for n in RESULTS:
    if n == "fix": continue
    mm, lo, hi, pw = paired(RESULTS[n]["pred"], RESULTS["fix"]["pred"]); sl[n] = mm
    print(f"  {n:>8} - fix: {mm:+.4f}  [{lo:+.4f}, {hi:+.4f}]  P(worse) = {pw:.2f}")

print("\n=== THE ANSWER ===")
pts = [(0.10, -sl.get("noise10", 0.0)), (0.25, -sl.get("noise25", 0.0))]
slope = sum(x*y for x, y in pts) / sum(x*x for x, _ in pts)
for frac, cost in pts:
    print(f"  {frac:.0%} label corruption costs {cost:+.4f} macro"
          f"   ({'resolved' if abs(cost) > 0.004 else 'below the noise floor at n=58'})")
print(f"  -> {slope:.4f} macro per unit of label accuracy, fit through the origin")
gain = 0.11 * slope
print(f"\n  The corrected lexicon disagrees with gold on 23% of cells. Recovering")
print(f"  half of that is ~11% better labels, worth roughly {gain:+.4f} macro.")
print("  -> premise HOLDS, week 3 earns its six days." if gain > 0.006 else
      "  -> marginal: an LLM labeler only if cheap, skip the full loop." if gain > 0.003 else
      "  -> premise FAILS at this resolution. Spend the six days on inference-time\n"
      "     work and the efficiency prize instead.")
print("\n  Caveat: n=58, and this extrapolates a large planted perturbation down to")
print("  a small real one. Order of magnitude, not a point estimate.")

llm = [n for n in RESULTS if n.startswith("llm_")]
if llm:
    print("\n=== a real alternative labeler, which beats the extrapolation ===")
    for n in llm:
        mm, lo, hi, pw = paired(RESULTS[n]["pred"], RESULTS["fix"]["pred"])
        print(f"  {n} - fix: {mm:+.4f}  [{lo:+.4f}, {hi:+.4f}]  P(worse) = {pw:.2f}")
    best = max(llm, key=lambda n: RESULTS[n]["macro"])
    dm = RESULTS[best]["macro"] - RESULTS["fix"]["macro"]
    print(f"\n  This is the direct measurement the slope above only estimates: a")
    print(f"  different labeler, trained through the same head, on the same")
    print(f"  features. Trust it over the extrapolation.")
    print("  -> a better labeler pays. Build on it." if dm > 0.004 else
          "  -> this labeler does not beat the lexicon here." if dm < -0.004 else
          "  -> no resolvable difference at n=58.")

print("\n=== per label ===")
cols = list(RESULTS)
short = {n: (n if len(n) <= 8 else n[:8]) for n in cols}   # keep columns aligned
print(f"{'label':<18}{'npos':>6}" + "".join(f"{short[n]:>9}" for n in cols))
for j, t in enumerate(LABELS):
    print(f"{t:<18}{int(yg[:, j].sum()):>6}" +
          "".join(f"{auc(yg[:, j], RESULTS[n]['pred'][:, j]):>9.3f}" for n in cols))

json.dump({"slope_per_unit_label_accuracy": float(slope),
           "projected_gain_from_better_labels": float(gain),
           "n_gold": int(is_gold.sum()), "n_train": int((~is_gold).sum()),
           "ckpt_macro": float(macro(yg, ckpt_pred)) if ckpt_pred is not None else None,
           "timing": TIMING,
           "arms": {n: {k: v for k, v in r.items() if k != "pred"} for n, r in RESULTS.items()}},
          open("/kaggle/working/premise_test.json", "w"), indent=1)
print("\nwrote /kaggle/working/premise_test.json and timing_probe.json")


## If it stopped early

The encode loop exits cleanly at `TIME_BUDGET_H` and every shard is already
written atomically, so nothing is lost. To continue:

1. Save this notebook's version (the output is kept automatically).
2. Add **this notebook's own output** as an input dataset to the next run.
3. Run again. `discover()` picks up the shards, `done` skips them, and encoding
   resumes where it stopped.

The gate section runs on whatever is cached, so you can read a result before the
full 4,407 are done. It needs at least 40 gold and 300 weakly-labelled studies,
and it asserts that rather than reporting nonsense.

## What to send back

`/kaggle/working/premise_test.json`, plus the printed output of the last two
cells. The number that decides week 3 is `slope_per_unit_label_accuracy`.
